# 🚗 Machine Learning Assignment 1 — Car Price Analysis

**Tasks Covered:**
- Task 1: Exploratory Data Analysis (EDA)
- Task 2: Data Preprocessing
- Task 3: Create Two Target Variables
- Task 4: Model 1 — Linear Regression
- Task 5: Model 2 — KNN Classification
- Task 6: Analysis and Discussion
- Task 7: Required Visualizations

---

## 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
print('All libraries imported successfully ✅')

---
## Task 1: Exploratory Data Analysis (EDA)

In [ ]:
# Load the dataset
df = pd.read_csv('car_price.csv')
print('Dataset loaded successfully!')
print(f'Shape: {df.shape}')
df.head(10)

### 1.1 — How many rows and columns does the dataset have?

In [ ]:
rows, cols = df.shape
print(f'📊 The dataset has {rows:,} rows and {cols} columns.')
print()
print('Column names:', list(df.columns))

### 1.2 — Which features are numerical? Which are categorical?

In [ ]:
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print('🔢 Numerical Features:', numerical_cols)
print()
print('🔤 Categorical Features:', categorical_cols)
print()
print(df.dtypes)

### 1.3 — Are there any missing values?

In [ ]:
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

print('❗ Missing Values Summary:')
print(missing_df)

# Visualize missing values
plt.figure(figsize=(10, 5))
sns.barplot(x=missing_df.index, y=missing_df['Missing Count'], palette='flare')
plt.title('Missing Values per Column', fontsize=14, fontweight='bold')
plt.xlabel('Columns')
plt.ylabel('Missing Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 1.4 — What does the distribution of car prices look like? (Task 7 — Price Distribution Histogram)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['price'].dropna(), bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Car Price Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price (£)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['price'].mean(), color='red', linestyle='--', label=f'Mean: £{df["price"].mean():,.0f}')
axes[0].axvline(df['price'].median(), color='orange', linestyle='--', label=f'Median: £{df["price"].median():,.0f}')
axes[0].legend()

# Log scale for skewness visualization
axes[1].hist(np.log1p(df['price'].dropna()), bins=60, color='darkcyan', edgecolor='white', alpha=0.85)
axes[1].set_title('Car Price Distribution (Log Scale)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Log(Price + 1)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print('📈 Price Statistics:')
print(df['price'].describe().round(2))

### 1.5 — Which features seem most related to price? (Task 7 — Correlation Heatmap)

In [ ]:
plt.figure(figsize=(9, 6))
corr_matrix = df[numerical_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap of Numerical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🔗 Correlation with Price (sorted):')
print(corr_matrix['price'].drop('price').sort_values(ascending=False))

> **Observation:** `year` is positively correlated with price (newer cars = higher price), while `mileage` is negatively correlated (more mileage = lower price). `engineSize` also shows a moderate positive correlation with price.

### Task 7 — Additional Plot 1: Price by Fuel Type (Boxplot)

In [ ]:
plt.figure(figsize=(11, 5))
order = df.groupby('fuelType')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='fuelType', y='price', order=order, palette='Set2')
plt.title('Price Distribution by Fuel Type', fontsize=13, fontweight='bold')
plt.xlabel('Fuel Type')
plt.ylabel('Price (£)')
plt.ylim(0, 80000)
plt.tight_layout()
plt.show()
print('\n💡 This plot shows how fuel type influences car pricing.\n'
      'Hybrid and diesel cars tend to have wider price ranges,\n'
      'reflecting diverse engine variants in the used-car market.')

### Task 7 — Additional Plot 2: Mileage vs Price (Scatter)

In [ ]:
sample = df.dropna(subset=['mileage', 'price']).sample(5000, random_state=42)
plt.figure(figsize=(10, 5))
plt.scatter(sample['mileage'], sample['price'], alpha=0.3, c='steelblue', s=10)
plt.title('Mileage vs Price (Sample of 5,000 Cars)', fontsize=13, fontweight='bold')
plt.xlabel('Mileage')
plt.ylabel('Price (£)')
plt.ylim(0, 80000)
plt.tight_layout()
plt.show()
print('\n💡 Clear negative trend: as mileage increases, price decreases.')

---
## Task 2: Data Preprocessing

In [ ]:
# Work on a copy
df_clean = df.copy()
print(f'Original shape: {df_clean.shape}')

### 2.1 — Handle Missing Values

> **Strategy & Justification:**
> All columns have ~5% missing values (≈3,621 rows out of 72,435). Since the missing rows are consistent across all columns — suggesting entire rows are blank — we **drop rows** where `price` (our target) is missing. Then we drop any remaining rows with NaN.
> - Dropping is justified because 5% data loss is acceptable and avoids imputation bias on the target variable.
> - Imputing `price` with mean/median would distort the training signal, which is dangerous for regression.

In [ ]:
# Drop rows where price (target) is missing
df_clean.dropna(subset=['price'], inplace=True)

# For numerical columns, fill remaining NaN with median (robust to outliers)
num_cols_to_fill = ['year', 'mileage', 'tax', 'mpg', 'engineSize']
for col in num_cols_to_fill:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

# For categorical columns, fill with mode
cat_cols_to_fill = ['model', 'transmission', 'fuelType', 'Make']
for col in cat_cols_to_fill:
    df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

# Final safety drop — remove any row that still has NaN anywhere
df_clean.dropna(inplace=True)
df_clean.reset_index(drop=True, inplace=True)
print(f'Shape after handling missing values: {df_clean.shape}')
print(f'Remaining missing values: {df_clean.isnull().sum().sum()}')

### 2.2 — Encode Categorical Columns

> **Strategy & Justification:**
> - `transmission` and `fuelType` → **Label Encoding**: these have a small number of categories and no natural ordering issue for tree/distance models. Label encoding keeps dimensionality low.
> - `Make` and `model` → **Label Encoding**: One-Hot would create hundreds of columns, inflating dimensionality and slowing KNN significantly. Label encoding is a practical trade-off here.

In [ ]:
le = LabelEncoder()
categorical_features = ['model', 'transmission', 'fuelType', 'Make']

for col in categorical_features:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

print('Categorical encoding done ✅')
print(df_clean[categorical_features].head())

### 2.3 — Detect and Handle Outliers (IQR Method)

In [ ]:
# Boxplot before outlier removal
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot(df_clean['price'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[0].set_title('Price — Before Outlier Removal', fontweight='bold')
axes[0].set_ylabel('Price (£)')

# IQR method
Q1 = df_clean['price'].quantile(0.25)
Q3 = df_clean['price'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f'Price IQR Range: £{lower_bound:,.0f} — £{upper_bound:,.0f}')
outliers = df_clean[(df_clean['price'] < lower_bound) | (df_clean['price'] > upper_bound)]
print(f'Outlier rows detected: {len(outliers):,}')

df_clean = df_clean[(df_clean['price'] >= lower_bound) & (df_clean['price'] <= upper_bound)]

axes[1].boxplot(df_clean['price'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightgreen'))
axes[1].set_title('Price — After Outlier Removal', fontweight='bold')
axes[1].set_ylabel('Price (£)')

plt.tight_layout()
plt.show()

print(f'\nShape after outlier removal: {df_clean.shape}')

---
## Task 3: Create Two Target Variables

### 3A — Regression Target: predict `price` directly

In [ ]:
feature_cols = ['year', 'mileage', 'tax', 'mpg', 'engineSize',
                'model', 'transmission', 'fuelType', 'Make']

X = df_clean[feature_cols]
y_reg = df_clean['price']  # Regression target

print('Regression target (y_reg) — first 5 values:')
print(y_reg.head())

### 3B — Classification Target: Cheap / Moderate / Expensive

> **Threshold Justification:**
> Based on the price distribution:
> - 33rd percentile ≈ £11,450 → **Cheap**: price < £11,450
> - 66th percentile ≈ £17,995 → **Moderate**: £11,450 ≤ price < £17,995
> - **Expensive**: price ≥ £17,995
>
> Using the 33rd and 66th percentiles ensures **balanced classes** (~equal thirds), which is critical for avoiding class imbalance in KNN classification.

In [ ]:
p33 = df_clean['price'].quantile(0.33)
p66 = df_clean['price'].quantile(0.66)
print(f'33rd percentile: £{p33:,.0f}')
print(f'66th percentile: £{p66:,.0f}')

def categorize_price(price):
    if price < p33:
        return 'Cheap'
    elif price < p66:
        return 'Moderate'
    else:
        return 'Expensive'

y_cls = df_clean['price'].apply(categorize_price)

print('\n📊 Cars per category:')
print(y_cls.value_counts())
print()

# Visualize category distribution
plt.figure(figsize=(7, 4))
counts = y_cls.value_counts()
colors = ['#4CAF50', '#FFC107', '#F44336']
bars = plt.bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'{val:,}', ha='center', fontweight='bold')
plt.title('Cars per Price Category', fontsize=13, fontweight='bold')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### 2.4 — Scale Numerical Features

> **Strategy:** We use **StandardScaler** (zero mean, unit variance). It is the preferred choice for KNN because KNN relies on distance — unscaled features with large ranges (like `mileage`) would dominate the distance calculation.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
print('Scaling done ✅')
print(X_scaled.describe().round(2))

---
## Task 4: Model 1 — Linear Regression

In [ ]:
# Train-test split (80/20)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_scaled, y_reg, test_size=0.2, random_state=42
)
print(f'Training set: {X_train_reg.shape}')
print(f'Test set:     {X_test_reg.shape}')

In [ ]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_reg, y_train_reg)

# Predict
y_pred_reg = lr_model.predict(X_test_reg)

# Evaluation
mae  = mean_absolute_error(y_test_reg, y_pred_reg)
mse  = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2   = r2_score(y_test_reg, y_pred_reg)

# Cross-validation R2
cv_r2 = cross_val_score(lr_model, X_scaled, y_reg, cv=5, scoring='r2')

print('=' * 45)
print('   LINEAR REGRESSION — Evaluation Metrics')
print('=' * 45)
print(f'  MAE  (Mean Absolute Error)   : £{mae:,.2f}')
print(f'  MSE  (Mean Squared Error)    : {mse:,.2f}')
print(f'  RMSE (Root Mean Sq. Error)   : £{rmse:,.2f}')
print(f'  R²   (Coefficient of Det.)   : {r2:.4f}')
print(f'  5-Fold CV R² (mean ± std)    : {cv_r2.mean():.4f} ± {cv_r2.std():.4f}')
print('=' * 45)

### Task 7 — Predicted vs Actual Scatter Plot (Regression)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Predicted vs Actual
axes[0].scatter(y_test_reg, y_pred_reg, alpha=0.3, s=10, color='steelblue')
axes[0].plot([y_test_reg.min(), y_test_reg.max()],
             [y_test_reg.min(), y_test_reg.max()],
             'r--', linewidth=1.5, label='Perfect Prediction')
axes[0].set_title('Predicted vs Actual — Linear Regression', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Actual Price (£)')
axes[0].set_ylabel('Predicted Price (£)')
axes[0].legend()

# Residuals
residuals = y_test_reg - y_pred_reg
axes[1].scatter(y_pred_reg, residuals, alpha=0.3, s=10, color='coral')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title('Residuals vs Fitted Values', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Fitted (Predicted) Price (£)')
axes[1].set_ylabel('Residuals')

plt.tight_layout()
plt.show()

# Feature importances (coefficients)
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': lr_model.coef_})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print('\n📊 Top Feature Coefficients:')
print(coef_df.to_string(index=False))

---
## Task 5: Model 2 — KNN Classification

In [ ]:
# Encode class labels
label_map = {'Cheap': 0, 'Moderate': 1, 'Expensive': 2}
y_cls_encoded = y_cls.map(label_map)

# Reset index to align with scaled X
y_cls_encoded = y_cls_encoded.reset_index(drop=True)
X_knn = X_scaled.reset_index(drop=True)

# 80/20 split
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_knn, y_cls_encoded, test_size=0.2, random_state=42, stratify=y_cls_encoded
)
print(f'Training set: {X_train_cls.shape}')
print(f'Test set:     {X_test_cls.shape}')

In [ ]:
# Grid Search with K-Fold Cross Validation
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'metric': ['euclidean', 'manhattan']
}

knn = KNeighborsClassifier()
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=kfold,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_cls, y_train_cls)

print(f'\n✅ Best Parameters: {grid_search.best_params_}')
print(f'✅ Best CV Accuracy: {grid_search.best_score_:.4f}')

In [ ]:
# Evaluate best model on test set
best_knn = grid_search.best_estimator_
y_pred_cls = best_knn.predict(X_test_cls)

acc  = accuracy_score(y_test_cls, y_pred_cls)
prec = precision_score(y_test_cls, y_pred_cls, average='weighted')
rec  = recall_score(y_test_cls, y_pred_cls, average='weighted')
f1   = f1_score(y_test_cls, y_pred_cls, average='weighted')

print('=' * 45)
print('   KNN CLASSIFIER — Evaluation Metrics')
print('=' * 45)
print(f'  Accuracy   : {acc:.4f}')
print(f'  Precision  : {prec:.4f}')
print(f'  Recall     : {rec:.4f}')
print(f'  F1-Score   : {f1:.4f}')
print('=' * 45)
print()
print('Classification Report:')
target_names = ['Cheap', 'Moderate', 'Expensive']
print(classification_report(y_test_cls, y_pred_cls, target_names=target_names))

### Task 7 — Confusion Matrix Heatmap

In [ ]:
cm = confusion_matrix(y_test_cls, y_pred_cls)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names,
            linewidths=0.5)
plt.title('KNN Confusion Matrix', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

print('\n📊 Confusion Matrix Explanation:')
print('- Diagonal cells = correct predictions')
print('- Off-diagonal cells = misclassifications')
print('- Most confusion occurs between adjacent categories (Cheap↔Moderate, Moderate↔Expensive)')
print('  which is expected since price thresholds are continuous.')

### KNN Grid Search Results Visualization

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
euclidean = results[results['param_metric'] == 'euclidean'].sort_values('param_n_neighbors')
manhattan  = results[results['param_metric'] == 'manhattan'].sort_values('param_n_neighbors')

plt.figure(figsize=(9, 5))
plt.plot(euclidean['param_n_neighbors'], euclidean['mean_test_score'],
         marker='o', label='Euclidean', color='steelblue')
plt.plot(manhattan['param_n_neighbors'], manhattan['mean_test_score'],
         marker='s', label='Manhattan', color='coral')
plt.title('KNN Grid Search — CV Accuracy by K and Distance Metric', fontsize=12, fontweight='bold')
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Mean CV Accuracy')
plt.xticks([3, 5, 7, 9, 11, 13, 15])
plt.legend()
plt.tight_layout()
plt.show()

---
## Task 6: Analysis and Discussion

### 6.1 — Model Comparison

> **Which model performed better?**
>
> - **Linear Regression** is evaluated with R², MAE, RMSE. An R² close to 1.0 means the model explains most of the variance in price. However, a high RMSE (e.g., £2,000–3,000) means the model is off by that amount on average — which may or may not be acceptable depending on the use case.
>
> - **KNN Classifier** is evaluated with accuracy, F1. A high accuracy (e.g., ≥80%) means the model correctly assigns price categories most of the time. Classification is inherently easier than regression because it discretizes the problem — the model only has to be right about which bin the price falls into, not the exact value.
>
> **Is classification easier than regression?**  
> Generally **yes** on this dataset. By collapsing continuous prices into 3 categories, we reduce the error space. The KNN model benefits from the fact that similar cars tend to cluster into the same price band.
>
> **Does converting price into categories lose important information?**  
> **Yes.** A car priced at £11,449 and one at £11,451 would be placed in different categories, even though their prices are almost identical. Discretization loses the fine-grained price signal that Linear Regression preserves. For applications like insurance pricing or dealer valuation, regression is superior.

### 6.2 — Sensitivity Analysis

In [ ]:
# --- Sensitivity 1: Remove the most correlated feature ---
most_corr = corr_matrix['price'].drop('price').abs().idxmax()
print(f'Most correlated feature with price: {most_corr}')

X_no_top = X_scaled.drop(columns=[most_corr])
X_tr, X_te, y_tr, y_te = train_test_split(X_no_top, y_reg, test_size=0.2, random_state=42)
lr_no_top = LinearRegression().fit(X_tr, y_tr)
r2_no_top = r2_score(y_te, lr_no_top.predict(X_te))

print(f'\nR² with all features:        {r2:.4f}')
print(f'R² without {most_corr}:  {r2_no_top:.4f}')
print(f'Performance drop:            {r2 - r2_no_top:.4f}')

In [ ]:
# --- Sensitivity 2: KNN without scaling ---
X_unscaled = X.reset_index(drop=True)
X_tr_u, X_te_u, y_tr_u, y_te_u = train_test_split(
    X_unscaled, y_cls_encoded, test_size=0.2, random_state=42, stratify=y_cls_encoded
)
best_params = grid_search.best_params_
knn_no_scale = KNeighborsClassifier(**best_params)
knn_no_scale.fit(X_tr_u, y_tr_u)
acc_no_scale = accuracy_score(y_te_u, knn_no_scale.predict(X_te_u))

print(f'KNN Accuracy WITH scaling:    {acc:.4f}')
print(f'KNN Accuracy WITHOUT scaling: {acc_no_scale:.4f}')
print(f'Accuracy drop without scaling: {acc - acc_no_scale:.4f}')
print('\n💡 Scaling has a significant impact on KNN because it relies on Euclidean/Manhattan distance.')

In [ ]:
# --- Sensitivity 3: Different threshold (25th/75th percentile) ---
p25 = df_clean['price'].quantile(0.25)
p75 = df_clean['price'].quantile(0.75)

def categorize_alt(price):
    if price < p25:
        return 0  # Cheap
    elif price < p75:
        return 1  # Moderate
    else:
        return 2  # Expensive

y_alt = df_clean['price'].apply(categorize_alt).reset_index(drop=True)
X_tr_a, X_te_a, y_tr_a, y_te_a = train_test_split(
    X_knn, y_alt, test_size=0.2, random_state=42, stratify=y_alt
)
knn_alt = KNeighborsClassifier(**best_params)
knn_alt.fit(X_tr_a, y_tr_a)
acc_alt = accuracy_score(y_te_a, knn_alt.predict(X_te_a))

print(f'KNN Accuracy with 33/66 thresholds: {acc:.4f}')
print(f'KNN Accuracy with 25/75 thresholds: {acc_alt:.4f}')
print('\n💡 Shifting thresholds creates an imbalanced class distribution (50% Moderate),'
      ' which slightly reduces performance.')

---
## ✅ Summary

| Metric | Linear Regression | KNN Classifier |
|--------|:-----------------:|:--------------:|
| Problem Type | Regression | Classification |
| Target | Exact Price (£) | Cheap / Moderate / Expensive |
| Primary Metric | R² Score | Accuracy / F1-Score |
| Scaling Required | No (but improves) | **Yes** (critical for KNN) |
| Hyperparameter Tuning | None | Grid Search (K, metric) |
| Interpretability | High (coefficients) | Low (black-box distances) |

---
*Machine Learning Assignment 1 — Car Price Analysis*